# Expected Returns Analytics
# 
# This notebook analyzes expected returns using the enhanced v2.0 analytics pipeline:
# - **Monte Carlo Simulation** — Probabilistic upside/downside distributions
# - **Price Target Achievement** — Probability-weighted expected returns by sector
# - **Kalman Filtered Targets** — Noise-reduced price target signals
# - **Analyst Sentiment Features** — Feature-level probability analytics
# - **Cross-Model Comparison** — MC vs Kalman vs Achievement model alignment
#
# Data sources: `analytics.monte_carlo_simulation`, `analytics.price_target_achievement`,
# `analytics.kalman_filtered_price_targets`, `analytics.earnings_probability_analysis`


## 1. Setup & Environment Configuration


In [1]:
import warnings
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")

# Configure database connection
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()

PLOTLY_TEMPLATE = "plotly_dark"
COLORS = px.colors.qualitative.Dark24

print("✅ Environment configured")

# ── Refactored analytics modules ─────────────────────────────────────
from probabilistic_ml_model.data_utils import (
    load_identifier_columns,
)
from probabilistic_ml_model import (
    PriceTargetAchievementModel,
    EarningsBeatProbabilityModel,
    EPSStreakAnalyzer,
    CreditRiskProbabilityModel,
    create_earnings_probability_dashboard,
)
from probabilistic_ml_model import (
    kalman_filter_price_target,
    kalman_momentum_filter,
    fit_gaussian_copula,
)
from probabilistic_ml_model import (
    fast_monte_carlo_simulation,
    get_optimization_status,
)
from probabilistic_ml_model import (
    create_analyst_upside_scatter,
    create_valuation_vs_growth_quadrant,
)

# --- InferenceData schema (ArviZ / xarray bridge) ---
try:
    from probabilistic_ml_model import (
        ARVIZ_AVAILABLE,
        build_monte_carlo_inference_data,
        summarize_inference_data,
    )
except ImportError:
    ARVIZ_AVAILABLE = False

# --- Probabilistic visualizations (ArviZ-backed) ---
try:
    from probabilistic_ml_model import (
        create_posterior_return_forest,
        create_beat_probability_posterior,
        create_ruin_probability_diagnostic,
        create_bayesian_category_ridge,
        create_tri_model_posterior_comparison,
    )
except ImportError:
    pass


✅ Environment configured


## 2. Data Acquisition


In [2]:
%%sql
SELECT * FROM analytics.monte_carlo_simulation

,ticker,name,region,country,exchange,sector,industry,last_price,pt_median,pt_spread,expected_upside_pct,upside_std,var_5_pct,prob_positive_upside,risk_reward_ratio
0,AGTHIA,Agthia Group PJSC,Africa / Middle East,AE,ADX,Consumer Staples,Food Products,3.83,4.8000,2.8000,33.887581,15.212635,11.009352,100.00,2.227594
1,TAQA,Abu Dhabi National Energy Company PJSC,Africa / Middle East,AE,ADX,Utilities,Multi-Utilities,2.91,3.0000,1.9500,-5.371855,14.027396,-30.615633,40.83,-0.382955
2,AIRARABIA,Air Arabia PJSC,Africa / Middle East,AE,DFM,Industrials,Passenger Airlines,5.50,4.4800,1.6400,-17.413829,6.094796,-27.475503,0.00,-2.857163
3,ADPORTS,Abu Dhabi Ports Company PJSC,Africa / Middle East,AE,ADX,Industrials,Transportation Infrastructure,5.02,7.0000,3.4000,35.524792,13.940557,11.289537,99.99,2.548305
4,ADNOCDIST,Abu Dhabi National Oil Company for Distributio...,Africa / Middle East,AE,ADX,Consumer Discretionary,Specialty Retail,4.07,4.5000,1.2800,16.128952,6.685957,6.614704,100.00,2.412363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,NPH,Northam Platinum Holdings Limited,Africa / Middle East,ZA,JSE,Materials,Metals and Mining,387.00,437.5000,144.0000,9.808239,7.651632,-3.977720,87.81,1.281849
5580,ZZD,Zeda Limited,Africa / Middle East,ZA,JSE,Industrials,Ground Transportation,12.69,16.1000,1.2800,24.443345,2.229041,20.267246,100.00,10.965857
5581,PMR,Premier Group Limited,Africa / Middle East,ZA,JSE,Consumer Staples,Food Products,181.26,208.0000,26.0000,12.159355,3.120915,6.490688,100.00,3.896086
5582,WBC,We Buy Cars Holdings Limited,Africa / Middle East,ZA,JSE,Consumer Discretionary,Specialty Retail,43.50,58.8800,13.8000,31.024949,6.699178,18.871082,100.00,4.631158


In [3]:
%%sql
SELECT * FROM analytics.price_target_achievement

,isin,ticker,name,region,country,trading_country,exchange,sector,industry,dividend_record_frequency,...,achievement_probability,upside_potential,price_target_spread_pct,analyst_conviction,eps_revision_momentum,analyst_rating_normalized,expected_return_prob_weighted,confidence_level,last_price,price_target_prob_weighted
0,AEA001901015,AGTHIA,Agthia Group PJSC,Africa / Middle East,AE,AE,ADX,Consumer Staples,Food Products,Interim Payment,...,0.53,25.326371,58.333333,85.714286,-0.080975,89.25,13.422977,Low,3.83,4.344100
1,AEA002401015,TAQA,Abu Dhabi National Energy Company PJSC,Africa / Middle East,AE,AE,ADX,Utilities,Multi-Utilities,Quarterly,...,0.60,3.092784,65.000000,25.000000,0.000055,33.25,1.855670,Low,2.91,2.964000
2,AEA002801016,EMSTEEL,EMSTEEL Building Materials PJSC,Africa / Middle East,AE,AE,ADX,Materials,Construction Materials,NaN,...,0.48,30.252101,0.000000,100.000000,0.000055,100.00,14.521008,Low,1.19,1.362800
3,AEA003001012,AIRARABIA,Air Arabia PJSC,Africa / Middle East,AE,AE,DFM,Industrials,Passenger Airlines,Annual,...,0.83,-18.545455,36.607143,12.500000,0.022885,50.00,-15.392727,Low,5.50,4.653400
4,AEA004601018,ADPORTS,Abu Dhabi Ports Company PJSC,Africa / Middle East,AE,AE,ADX,Industrials,Transportation Infrastructure,NaN,...,0.37,39.442231,48.571429,75.000000,0.043085,81.25,14.593625,Low,5.02,5.752600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6400,ZAE000315768,ZZD,Zeda Limited,Africa / Middle East,ZA,ZA,JSE,Industrials,Ground Transportation,Final Payment,...,0.61,26.871552,7.950311,66.666667,0.103670,100.00,16.391647,High,12.69,14.770100
6401,ZAE000320321,PMR,Premier Group Limited,Africa / Middle East,ZA,ZA,JSE,Consumer Staples,Food Products,Interim Payment,...,0.90,14.752290,12.500000,100.000000,0.115470,91.75,13.277061,High,181.26,205.326000
6402,ZAE000322095,NPK,Nampak Limited,Africa / Middle East,ZA,ZA,JSE,Materials,Containers and Packaging,NaN,...,0.28,45.636911,0.000000,0.000000,0.000480,-25.00,12.778335,Low,498.50,562.200000
6403,ZAE000332789,WBC,We Buy Cars Holdings Limited,Africa / Middle East,ZA,ZA,JSE,Consumer Discretionary,Specialty Retail,Final Payment,...,0.23,35.356322,23.437500,0.000000,0.006310,50.00,8.131954,Medium,43.50,47.037400


In [4]:
%%sql
SELECT * FROM analytics.kalman_filtered_price_targets

,ticker,name,country,exchange,sector,industry,kalman_estimate,kalman_variance,kalman_gain,signal_strength,original_price,original_target,filtered_upside
0,AGTHIA,Agthia Group PJSC,AE,ADX,Consumer Staples,Food Products,5.114456,0.090909,0.909092,10.99999,3.83,5.2429,33.536703
1,TAQA,Abu Dhabi National Energy Company PJSC,AE,ADX,Utilities,Multi-Utilities,2.764545,0.090909,0.909092,10.99999,2.91,2.7500,-4.998443
2,EMSTEEL,EMSTEEL Building Materials PJSC,AE,ADX,Materials,Construction Materials,1.517273,0.090909,0.909092,10.99999,1.19,1.5500,27.501935
3,AIRARABIA,Air Arabia PJSC,AE,DFM,Industrials,Passenger Airlines,4.613636,0.090909,0.909092,10.99999,5.50,4.5250,-16.115717
4,ADPORTS,Abu Dhabi Ports Company PJSC,AE,ADX,Industrials,Transportation Infrastructure,6.581365,0.090909,0.909092,10.99999,5.02,6.7375,31.102890
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6398,ZZD,Zeda Limited,ZA,JSE,Industrials,Ground Transportation,15.511184,0.090909,0.909092,10.99999,12.69,15.7933,22.231555
6399,PMR,Premier Group Limited,ZA,JSE,Consumer Staples,Food Products,201.326655,0.090909,0.909092,10.99999,181.26,203.3333,11.070647
6400,NPK,Nampak Limited,ZA,JSE,Materials,Containers and Packaging,705.318370,0.090909,0.909092,10.99999,498.50,726.0000,41.488138
6401,WBC,We Buy Cars Holdings Limited,ZA,JSE,Consumer Discretionary,Specialty Retail,55.514556,0.090909,0.909092,10.99999,43.50,56.7160,27.619670


In [5]:
%%sql
SELECT * FROM public.vw_features_analyst_sentiment
ORDER BY next_earnings ASC

,isin,ticker,name,region,country,trading_country,exchange,sector,industry,dividend_record_frequency,...,pt_median_momentum_1m,pt_median_momentum_3m,pt_acceleration_short,pt_acceleration_long,pt_consensus_convergence,analyst_coverage_change_1m,analyst_coverage_change_3m,analyst_coverage_change_1y,pt_vs_price_momentum,analyst_coverage_trend
0,SE0000407991,SVEDB,Svedbergs Group AB (publ),Europe,SE,SE,OM,Industrials,Building Products,Interim Payment,...,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN
1,FR001400TL40,ALHG,Louis Hachette Group S.A.,Europe,FR,FR,ENXTPA,Consumer Staples,Media,Annual,...,0.043478,0.066667,-0.007750,-0.023906,0.091667,0,1,1,-0.125745,0.200000
2,SE0012116390,VPLAYB,Viaplay Group AB (publ),Europe,SE,SE,OM,Consumer Staples,Media,NaN,...,0.000000,NaN,NaN,NaN,NaN,0,1,-1,NaN,0.600000
3,SE0021020716,AAC,AAC Clyde Space AB (publ),Europe,SE,SE,OM,Industrials,Aerospace and Defense,NaN,...,0.054670,0.042793,0.011877,NaN,-0.116631,0,1,2,-0.256838,0.425000
4,NL0012866412,BESI,BE Semiconductor Industries N.V.,Europe,NL,NL,ENXTAM,Information Technology,Semiconductors and Semiconductor Equipment,Annual,...,0.127596,0.225806,-0.102952,-0.051644,-0.158065,1,3,2,-0.140704,0.095652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6401,AU000000ANN9,ANN,Ansell Limited,Asia / Pacific,AU,AU,ASX,Health Care,Health Care Equipment and Supplies,Interim Payment,...,-0.030555,-0.038091,0.008475,0.005576,0.089370,1,1,-1,0.043574,0.075000
6402,NL0015000D50,NXFIL,NX Filtration N.V.,Europe,NL,NL,ENXTAM,Industrials,Machinery,NaN,...,-0.140845,-0.197368,0.123520,0.056311,-0.221311,0,0,-1,-0.037281,-0.062500
6403,AT000000STR1,STR,Strabag SE,Europe,AT,AT,WBAG,Industrials,Construction and Engineering,Annual,...,0.151399,0.151399,0.000000,-0.861065,0.015785,0,0,0,-0.095330,0.000000
6404,FR0004027068,ALLAN,Lanson-BCC,Europe,FR,FR,ENXTPA,Consumer Staples,Beverages,Annual,...,-0.048193,-0.048193,0.000000,0.022395,-0.109806,0,0,0,0.062482,0.000000


## 3. Data Overview & Quality Checks


In [6]:
# Rename the DataSpell-imported variables to convenient names
# (Adjust variable names if DataSpell assigns different ones)
try:
    mc = mc_sim.copy()
except NameError:
    print("⚠️ Run the data_input cells above first")

try:
    pt = pt_a.copy()
except NameError:
    print("⚠️ Run the price_target_achievement data_input cell first")

try:
    kal = pt_kal.copy()
except NameError:
    print("⚠️ Run the kalman_filtered_price_targets data_input cell first")

print(f"Monte Carlo Simulation:        {mc.shape[0]:,} stocks × {mc.shape[1]} cols")
print(f"Price Target Achievement:      {pt.shape[0]:,} stocks × {pt.shape[1]} cols")
print(f"Kalman Filtered Targets:       {kal.shape[0]:,} stocks × {kal.shape[1]} cols")

# Summary statistics for core return metrics
display(mc[["expected_upside_pct", "var_5_pct", "prob_positive_upside", "risk_reward_ratio"]].describe().round(2))


Monte Carlo Simulation:        5,584 stocks × 15 cols
Price Target Achievement:      6,405 stocks × 41 cols
Kalman Filtered Targets:       6,403 stocks × 13 cols


,expected_upside_pct,var_5_pct,prob_positive_upside,risk_reward_ratio
count,5584.00,5584.00,5584.00,5584.00
mean,22.44,3.01,74.83,2.79
std,40.47,30.88,32.50,12.84
min,-76.96,-82.68,0.00,-215.94
25%,0.93,-14.65,55.18,0.11
50%,13.46,-1.24,93.02,1.50
75%,32.64,15.33,100.00,3.24
max,751.49,335.01,100.00,539.79


## 4. Monte Carlo Simulation Analysis


### 4.1 Expected Upside Distribution


In [7]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Expected Upside Distribution", "Probability of Positive Return"),
    vertical_spacing=0.12,
)

# Clip extreme outliers for better visualization
upside_clipped = mc["expected_upside_pct"].clip(-100, 300)

fig.add_trace(
    go.Histogram(
        x=upside_clipped,
        nbinsx=80,
        marker_color=COLORS[0],
        opacity=0.75,
        name="Expected Upside %",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(
    x=mc["expected_upside_pct"].median(),
    line_dash="dot", line_color="green",
    annotation_text=f"Median: {mc['expected_upside_pct'].median():.1f}%",
    row=1, col=1,
)

# Probability of positive return - pie chart
prob_bins = pd.cut(mc["prob_positive_upside"], bins=[0, 25, 50, 75, 100],
                   labels=["0-25%", "25-50%", "50-75%", "75-100%"])
prob_counts = prob_bins.value_counts().sort_index()
fig.add_trace(
    go.Bar(
        x=prob_counts.index.astype(str),
        y=prob_counts.values,
        marker_color=[COLORS[3], COLORS[1], COLORS[0], COLORS[2]],
        name="Stock Count",
    ),
    row=2, col=1,
)

fig.update_layout(
    title="Monte Carlo Simulation: Return Distribution Overview",
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1000,
    showlegend=True,
)
fig.update_xaxes(title_text="Expected Upside (%)", row=1, col=1)
fig.update_xaxes(title_text="Probability of Positive Return", row=2, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=2, col=1)
fig.show()


### 4.2 Risk-Reward by Industry


In [8]:
# Sector-level aggregation
mc_sector = (
    mc.groupby("industry")
    .agg(
        mean_upside=("expected_upside_pct", "mean"),
        median_upside=("expected_upside_pct", "median"),
        mean_var5=("var_5_pct", "mean"),
        mean_prob_positive=("prob_positive_upside", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_upside", ascending=False)
)

fig = px.scatter(
    mc_sector,
    x="mean_var5",
    y="mean_upside",
    size="count",
    color="industry",
    hover_name="industry",
    hover_data={"mean_prob_positive": ":.1f", "count": True},
    title="Industry Risk-Reward: Expected Upside vs Value-at-Risk (5%)",
    labels={
        "mean_var5": "Mean VaR 5% (%)",
        "mean_upside": "Mean Expected Upside (%)",
        "count": "# Stocks",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()


### 4.3 Top Opportunities — Highest Risk-Reward Ratio (Positive Upside)


In [9]:
mc_positive = mc[mc["prob_positive_upside"] >= 75].nlargest(50, "risk_reward_ratio")

fig = px.bar(
    mc_positive,
    x="ticker",
    y="expected_upside_pct",
    color="industry",
    hover_data=["name", "prob_positive_upside", "risk_reward_ratio"],
    title="Top 50 Opportunities: Highest Risk-Reward (≥75% Prob Positive)",
    labels={"expected_upside_pct": "Expected Upside (%)", "ticker": "Ticker"},
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 5. Price Target Achievement Analysis


### 5.1 Achievement Probability Distribution by Confidence Level


In [10]:
fig = px.violin(
    pt,
    x="confidence_level",
    y="achievement_probability",
    color="confidence_level",
    box=True,
    points="outliers",
    title="Price Target Achievement Probability by Confidence Level",
    labels={
        "achievement_probability": "Achievement Probability",
        "confidence_level": "Confidence Level",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=450,
)
fig.show()


### 5.2 Probability-Weighted Expected Return by Sector


In [11]:
pt_sector = (
    pt.groupby("industry")
    .agg(
        mean_expected_return=("expected_return_prob_weighted", "mean"),
        median_expected_return=("expected_return_prob_weighted", "median"),
        mean_achievement_prob=("achievement_probability", "mean"),
        mean_conviction=("analyst_conviction", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_expected_return", ascending=True)
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=pt_sector["industry"],
        x=pt_sector["mean_expected_return"],
        orientation="h",
        marker_color=[
            COLORS[2] if v >= 0 else COLORS[3]
            for v in pt_sector["mean_expected_return"]
        ],
        text=pt_sector["mean_expected_return"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean Prob-Weighted Return: %{x:.2f}%<br>"
            "Avg Achievement Prob: %{customdata[0]:.0%}<br>"
            "Avg Conviction: %{customdata[1]:.1f}<br>"
            "Stocks: %{customdata[2]}"
        ),
        customdata=pt_sector[["mean_achievement_prob", "mean_conviction", "count"]].values,
    )
)
fig.update_layout(
    title="Probability-Weighted Expected Return by Industry",
    xaxis_title="Mean Expected Return (%)",
    template=PLOTLY_TEMPLATE,
    height=1100,
    margin=dict(l=350),
)
fig.show()


### 5.3 Conviction vs Upside Potential Scatter


In [12]:
sample_pt = pt.dropna(subset=["analyst_conviction"]).sample(min(2000, len(pt)), random_state=42)
fig = px.scatter(
    sample_pt,
    x="expected_return_prob_weighted",
    y="upside_potential",
    color="achievement_probability",
    size="analyst_conviction",
    hover_name="ticker",
    hover_data=["name", "sector", "industry", "expected_return_prob_weighted", "confidence_level"],
    title="Analyst Conviction vs Upside Potential",
    labels={
        "analyst_conviction": "Analyst Conviction (%)",
        "upside_potential": "Upside Potential (%)",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)
fig.show()


## 6. Kalman Filtered Price Target Analysis


### 6.1 Kalman Filtered vs Original Upside


In [14]:
# Pre-compute the column on the full DataFrame
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100

# Sample AFTER the column exists
kal_sample = kal.sample(min(2000, len(kal)), random_state=42).copy()

# Apply signed log1p transform for axis-aligned visualization
kal_sample["filtered_upside_log"] = np.sign(kal_sample["filtered_upside"]) * np.log1p(
    np.abs(kal_sample["filtered_upside"]))
kal_sample["raw_upside_log"] = np.sign(kal_sample["raw_upside"]) * np.log1p(np.abs(kal_sample["raw_upside"]))

fig = px.scatter(
    kal_sample,
    x="filtered_upside_log",
    y="raw_upside_log",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "kalman_estimate", "original_target", "original_price", "filtered_upside", "raw_upside"],
    title="Kalman-Filtered Upside vs Raw Analyst Upside (Log-Transformed Axes)",
    labels={
        "filtered_upside_log": "Kalman Filtered Upside — sign(x)·log₁ₚ(|x|)",
        "raw_upside_log": "Raw Analyst Upside — sign(x)·log₁ₚ(|x|)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)

# Add diagonal reference line on the log-transformed scale
log_max = max(
    kal_sample["filtered_upside_log"].abs().quantile(0.99),
    kal_sample["raw_upside_log"].abs().quantile(0.99),
)
fig.add_shape(
    type="line", x0=-log_max, y0=-log_max, x1=log_max, y1=log_max,
    line=dict(color="gray", dash="dash", width=1),
)
fig.show()


### 6.2 Signal Strength Distribution by Sector


In [15]:
fig = px.box(
    kal,
    x="industry",
    y="filtered_upside",
    color="industry",
    title="Kalman-Filtered Upside Distribution by Sector",
    labels={
        "filtered_upside": "Filtered Upside (%)",
        "industry": "",
    },
    template=PLOTLY_TEMPLATE,
    height=1000,
)
fig.update_layout(
    xaxis_tickangle=-65,
    showlegend=False,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
fig.show()


### 6.3 Kalman Noise Reduction Effectiveness


In [16]:
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100
kal["noise_reduction"] = abs(kal["raw_upside"] - kal["filtered_upside"])

noise_by_sector = (
    kal.groupby("industry")
    .agg(
        mean_noise_reduction=("noise_reduction", "mean"),
        median_raw_upside=("raw_upside", "median"),
        median_filtered_upside=("filtered_upside", "median"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_noise_reduction", ascending=False)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_raw_upside"],
    name="Raw Median Upside",
    marker_color=COLORS[1],
    opacity=0.7,
))
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_filtered_upside"],
    name="Kalman-Filtered Median Upside",
    marker_color=COLORS[0],
))
fig.update_layout(
    title="Kalman Filter Impact: Raw vs Filtered Median Upside by Sector",
    yaxis_title="Median Upside (%)",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    height=1000,
    xaxis_tickangle=-85,
)
fig.show()


### 6.4 Export Kalman-Filtered Targets


## 7. Cross-Model Comparison


### 7.1 MC Expected Upside vs Kalman Filtered Upside


In [18]:
# Merge Monte Carlo and Kalman results
mc_kal = mc.merge(
    kal[["ticker", "country", "exchange", "filtered_upside", "kalman_estimate", "original_price", "original_target"]],
    on="ticker",
    how="inner",
)

fig = px.scatter(
    mc_kal.sample(min(2000, len(mc_kal)), random_state=42),
    x="expected_upside_pct",
    y="filtered_upside",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "original_price", "kalman_estimate", "original_target", "prob_positive_upside"],
    title="Monte Carlo vs Kalman-Filtered Expected Returns",
    labels={
        "expected_upside_pct": "MC Expected Upside (%)",
        "filtered_upside": "Kalman Filtered Upside (%)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.5,
)
# Diagonal reference
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=200, y1=200,
    line=dict(color="red", dash="dash", width=2),
)
fig.show()

In [19]:

# Correlation summary
corr = mc_kal[["expected_upside_pct", "filtered_upside"]].corr().iloc[0, 1]
print(f"📊 MC ↔ Kalman correlation: {corr:.3f}")


📊 MC ↔ Kalman correlation: 0.979


### 7.2 Tri-Model Alignment: MC + Kalman + Achievement


In [20]:
# Merge all three models
tri = (
    mc[["ticker", "name", "sector", "industry", "expected_upside_pct", "prob_positive_upside"]]
    .merge(
        kal[["ticker", "filtered_upside"]],
        on="ticker",
        how="inner",
    )
    .merge(
        pt[["ticker", "expected_return_prob_weighted", "achievement_probability", "confidence_level"]],
        on="ticker",
        how="inner",
    )
)

# Agreement score: all three models agree on direction
tri["mc_bullish"] = tri["expected_upside_pct"] > 0
tri["kal_bullish"] = tri["filtered_upside"] > 0
tri["pt_bullish"] = tri["expected_return_prob_weighted"] > 0
tri["agreement_score"] = (
        tri["mc_bullish"].astype(int)
        + tri["kal_bullish"].astype(int)
        + tri["pt_bullish"].astype(int)
)
tri["signal"] = tri["agreement_score"].map(
    {0: "Strong Bearish (0/3)", 1: "Bearish (1/3)", 2: "Bullish (2/3)", 3: "Strong Bullish (3/3)"}
)

fig = px.histogram(
    tri,
    x="signal",
    color="signal",
    title="Tri-Model Signal Agreement (MC + Kalman + Achievement)",
    labels={"signal": "Model Agreement", "count": "Number of Stocks"},
    color_discrete_map={
        "Strong Bearish (0/3)": COLORS[3],
        "Bearish (1/3)": COLORS[1],
        "Bullish (2/3)": COLORS[0],
        "Strong Bullish (3/3)": COLORS[2],
    },
    category_orders={"signal": [
        "Strong Bearish (0/3)", "Bearish (1/3)",
        "Bullish (2/3)", "Strong Bullish (3/3)",
    ]},
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(showlegend=False)
fig.show()

In [21]:

print(f"\n📊 Model Agreement Summary:")
print(tri["signal"].value_counts().to_string())



📊 Model Agreement Summary:
signal
Strong Bullish (3/3)    4106
Strong Bearish (0/3)    1013
Bullish (2/3)            247
Bearish (1/3)            218


### 7.3 Strong Consensus Picks — All 3 Models Bullish, High Confidence


In [22]:
strong_consensus = (
    tri[
        (tri["agreement_score"] == 3)
        & (tri["prob_positive_upside"] >= 55)
        & (tri["achievement_probability"] >= 0.6)
        ]
    .nlargest(50, "expected_upside_pct")
)

if len(strong_consensus) > 0:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_upside_pct"],
        name="MC Expected Upside",
        marker_color=COLORS[0],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["filtered_upside"],
        name="Kalman Filtered Upside",
        marker_color=COLORS[1],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_return_prob_weighted"],
        name="Prob-Weighted Return",
        marker_color=COLORS[2],
    ))
    fig.update_layout(
        title=f"Top {len(strong_consensus)} Strong Consensus Picks (All 3 Models Bullish)",
        yaxis_title="Expected Return (%)",
        barmode="group",
        template=PLOTLY_TEMPLATE,
        height=500,
        xaxis_tickangle=-45,
    )
    fig.show()

    display(
        strong_consensus[["ticker", "name", "sector", "industry", "expected_upside_pct",
                          "filtered_upside", "expected_return_prob_weighted",
                          "prob_positive_upside", "achievement_probability", "confidence_level"]]
        .reset_index(drop=True)
    )
else:
    print("No stocks meet the strong consensus criteria.")


,ticker,name,sector,industry,expected_upside_pct,filtered_upside,expected_return_prob_weighted,prob_positive_upside,achievement_probability,confidence_level
0,TLO,Talon Metals Corp.,Materials,Metals and Mining,339.860312,234.725606,10.132939,100.00,0.70,Low
1,NVM,Novem Group S.A.,Consumer Discretionary,Automobile Components,255.429454,234.196480,9.695364,99.88,0.61,Low
2,TLEVISACPO,Grupo Televisa S.A.B.,Communication Services,Diversified Telecommunication Services,117.398247,67.306029,1.367347,96.41,0.67,Low
3,SFOR,S4 Capital plc,Consumer Staples,Media,92.201340,50.702525,11.090909,99.56,0.61,Low
4,BHIA3,Grupo Casas Bahia S.A.,Consumer Discretionary,Specialty Retail,61.787092,26.880121,2.033223,93.92,0.68,Low
5,IBRX,ImmunityBio Inc.,Health Care,Biotechnology,52.130286,32.392924,3.000000,91.33,0.87,Low
6,FRVIA,Forvia SE,Consumer Discretionary,Automobile Components,51.656556,9.232901,0.897325,87.50,0.73,Low
7,ORBIA,Orbia Advance Corporation S.A.B. de C.V.,Materials,Chemicals,46.377658,24.141294,4.229202,84.39,0.73,Low
8,QXO,QXO Inc.,Industrials,Trading Companies and Distributors,37.820598,23.468734,10.826938,100.00,0.62,Low
9,AMOB3,Automob Participações S.A.,Consumer Discretionary,Specialty Retail,35.571350,32.353413,17.142857,100.00,0.60,High


### 7.4 Quad-Model Alignment: MC + Kalman + Achievement + EPS Streak


In [23]:
# Quad-model agreement (4/4): MC + Kalman + PT Achievement + Beat Probability
BEAT_BULLISH_THRESHOLD = 0.6

_has_beat_data = (
        not beat_results.empty
        and 'ticker' in beat_results.columns
        and 'posterior_beat_prob' in beat_results.columns
)

if _has_beat_data and 'ticker' in tri.columns:
    beat_slim = beat_results[['ticker', 'posterior_beat_prob']].rename(
        columns={'posterior_beat_prob': 'beat_prob'}
    )
    quad = tri.merge(beat_slim, on='ticker', how='inner')

    if quad.empty:
        print("⚠️ No overlapping tickers between tri-model and beat_results")
    else:
        quad['beat_bullish_flag'] = (quad['beat_prob'] >= BEAT_BULLISH_THRESHOLD).astype(int)
        quad['quad_agreement'] = (
                quad['mc_bullish'].astype(int)
                + quad['kalman_bullish'].astype(int)
                + quad['pt_bullish'].astype(int)
                + quad['beat_bullish_flag']
        )

        fig = px.histogram(
            quad,
            x='quad_agreement',
            nbins=5,
            title='📊 Quad-Model Agreement Distribution (MC + Kalman + PT + Beat)',
            labels={'quad_agreement': 'Models Agreeing (out of 4)'},
            color_discrete_sequence=['#2E91E5'],
            template=PLOTLY_TEMPLATE,
            height=420,
        )
        fig.update_xaxes(dtick=1)
        fig.show()

        full_consensus = (quad['quad_agreement'] == 4).sum()
        no_consensus = (quad['quad_agreement'] == 0).sum()
        print(f"📊 Full consensus (4/4): {full_consensus} stocks")
        print(f"📊 No consensus  (0/4): {no_consensus} stocks")
        print(f"📊 Total quad-model coverage: {len(quad):,} stocks")
else:
    reasons = []
    if beat_results.empty:
        reasons.append("beat_results is empty")
    elif 'ticker' not in beat_results.columns:
        reasons.append("beat_results missing 'ticker' column")
    elif 'posterior_beat_prob' not in beat_results.columns:
        reasons.append("beat_results missing 'posterior_beat_prob' column")
    print(f"⚠️ Insufficient data for quad-model agreement ({'; '.join(reasons)})")

⚠️ Insufficient data for quad-model agreement (beat_results is empty)


### 7.5 Cross-Model Dependency Structure (Gaussian Copula)


In [24]:
# Measure tail dependence between MC and Kalman return signals
if len(mc_kal) > 50:
    copula_result = fit_gaussian_copula(mc_kal, features=['expected_upside_pct', 'filtered_upside'])
    if copula_result:
        print(f"📊 MC ↔ Kalman Dependency Analysis:")
        print(f"   Correlation matrix:\n{copula_result.get('correlation_matrix', 'N/A')}")
        print(f"   Tail dependence: {copula_result.get('tail_dependence', 'N/A')}")


📊 MC ↔ Kalman Dependency Analysis:
   Correlation matrix:
[[1.         0.98634408]
 [0.98634408 1.        ]]
   Tail dependence: {'lower': array([[1.        , 0.91397849],
       [0.91397849, 1.        ]]), 'upper': array([[1.        , 0.91756272],
       [0.91756272, 1.        ]])}


## 8. Sector Expected Returns Heatmap


In [25]:
# Aggregate all return metrics by sector
sector_returns = (
    tri.groupby("industry")
    .agg(
        mc_mean=("expected_upside_pct", "mean"),
        mc_median=("expected_upside_pct", "median"),
        kalman_mean=("filtered_upside", "mean"),
        kalman_median=("filtered_upside", "median"),
        pt_mean=("expected_return_prob_weighted", "mean"),
        pt_median=("expected_return_prob_weighted", "median"),
        pct_bullish=("agreement_score", lambda x: (x == 3).mean() * 100),
        count=("ticker", "count"),
    )
    .reset_index()
)

heatmap_data = sector_returns.set_index("industry")[
    ["mc_mean", "mc_median", "kalman_mean", "kalman_median", "pt_mean", "pt_median", "pct_bullish"]
].rename(columns={
    "mc_mean": "MC Mean",
    "mc_median": "MC Median",
    "kalman_mean": "Kalman Mean",
    "kalman_median": "Kalman Median",
    "pt_mean": "Achiev. Mean",
    "pt_median": "Achiev. Median",
    "pct_bullish": "% All Bullish",
})

fig = px.imshow(
    heatmap_data.round(1),
    color_continuous_scale="RdYlGn",
    text_auto=True,
    aspect="auto",
    title="Industry Expected Returns Heatmap (All Models)",
    labels={"color": "Value"},
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=1250,
)
fig.show()


## 9. VaR & Tail Risk Analysis


In [26]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("VaR 5% Distribution", "VaR 5% vs Expected Upside"),
    vertical_spacing=0.12,
)

# VaR distribution
var_clipped = mc["var_5_pct"].clip(-150, 300)
fig.add_trace(
    go.Histogram(
        x=var_clipped,
        nbinsx=80,
        marker_color=COLORS[3],
        opacity=0.75,
        name="VaR 5%",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="blue", row=1, col=1)

# VaR vs Expected Upside (sampled for performance)
sample = mc.sample(min(2000, len(mc)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["var_5_pct"],
        y=sample["expected_upside_pct"],
        mode="markers",
        marker=dict(
            size=4,
            color=sample["prob_positive_upside"],
            colorscale="RdYlGn",
            colorbar=dict(title="P(+)"),
            opacity=0.5,
        ),
        text=sample["name"],
        hovertemplate="%{text}<br>VaR 5%%: %{x:.1f}%<br>Expected Upside: %{y:.1f}%<extra></extra>",
        name="Stocks",
    ),
    row=2, col=1,
)
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=300, y1=300,
    line=dict(color="gray", dash="dash", width=1),
    row=2, col=1,
)

fig.update_layout(
    title="Value-at-Risk (5%) Analysis",
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1000,
    showlegend=False,
)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=1)
fig.update_xaxes(title_text="VaR 5% (%)", row=2, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Expected Upside (%)", row=2, col=1)
fig.show()


In [27]:
identifier_cols = load_identifier_columns()


def _reorder_with_identifiers(result_df: pd.DataFrame) -> pd.DataFrame:
    id_cols = [c for c in identifier_cols if c in result_df.columns]
    other_cols = [c for c in result_df.columns if c not in id_cols]
    return result_df[id_cols + other_cols]


## 9.5c Inline Fast MC Simulation (Numba-accelerated)


In [29]:
# Optional: Inline fast MC simulation (Numba-accelerated)
opt_status = get_optimization_status()
print(f"🔧 Numba available: {opt_status.get('numba_available')}")

# mc_inline = fast_monte_carlo_simulation(source_df, n_simulations=10000)


🔧 Numba available: True


## 9.5 InferenceData Schema Integration

Build ArviZ-compatible InferenceData from Monte Carlo simulation results
for standardised posterior analysis, diagnostics, and NetCDF export.


In [30]:
# Build InferenceData from Monte Carlo simulation results
if ARVIZ_AVAILABLE and 'mc' in dir() and len(mc) > 0:
    try:
        idata_mc = build_monte_carlo_inference_data(
            mc, mc, n_simulations=10000,
        )
        mc_summary = summarize_inference_data(idata_mc)
        print(f"✅ InferenceData built: {mc_summary.get('groups', [])}")
        print(f"   Draws: {mc_summary.get('n_draws', 0)}, Equities: {mc_summary.get('n_equities', 0)}")
        if mc_summary.get('r_hat'):
            for var, rhat_val in mc_summary['r_hat'].items():
                print(f"   R-hat ({var}): {rhat_val:.4f}")
    except Exception as e:
        print(f"⚠️ InferenceData build failed: {e}")
else:
    print('⚠️ ArviZ not available or no MC data')


✅ InferenceData built: ['posterior_predictive', 'observed_data', 'constant_data']
   Draws: 10000, Equities: 5584


## 9.6 Probabilistic Visualizations

Generate ArviZ-backed probabilistic charts from Monte Carlo and Bayesian results.


In [31]:
# Posterior return forest from Monte Carlo results
if 'mc' in dir() and len(mc) > 0:
    fig = create_posterior_return_forest(mc, top_n=25)
    fig.show()
    fig.write_html('outputs/analytics/posterior_return_forest.html')
    print('✓ Saved posterior_return_forest.html')


✓ Saved posterior_return_forest.html


In [32]:
# Tri-model posterior comparison (requires tri-model alignment DataFrame)
tri_cols = {'name', 'expected_upside_pct', 'filtered_upside', 'expected_return_prob_weighted'}
if 'strong_consensus' in dir() and tri_cols.issubset(strong_consensus.columns):
    fig = create_tri_model_posterior_comparison(strong_consensus, top_n=25)
    fig.show()
    fig.write_html('outputs/analytics/tri_model_posterior_comparison.html')
    print('✓ Saved tri_model_posterior_comparison.html')
else:
    print('⚠️ Tri-model columns not available — skipping')


✓ Saved tri_model_posterior_comparison.html


## 10. Summary Statistics


In [33]:
summary = {
    "Monte Carlo": {
        "Stocks Analyzed": len(mc),
        "Mean Expected Upside (%)": mc["expected_upside_pct"].mean().round(2),
        "Median Expected Upside (%)": mc["expected_upside_pct"].median().round(2),
        "% Stocks with Positive Upside": (mc["expected_upside_pct"] > 0).mean() * 100,
        "Mean Prob Positive (%)": mc["prob_positive_upside"].mean().round(1),
    },
    "Price Target Achievement": {
        "Stocks Analyzed": len(pt),
        "Mean Prob-Weighted Return (%)": pt["expected_return_prob_weighted"].mean().round(2),
        "Mean Achievement Prob": pt["achievement_probability"].mean().round(3),
        "High Confidence Count": (pt["confidence_level"] == "High").sum(),
        "Mean Analyst Conviction (%)": pt["analyst_conviction"].mean().round(1),
    },
    "Kalman Filter": {
        "Stocks Analyzed": len(kal),
        "Mean Filtered Upside (%)": kal["filtered_upside"].mean().round(2),
        "Median Filtered Upside (%)": kal["filtered_upside"].median().round(2),
        "Mean Signal Strength": kal["signal_strength"].mean().round(2),
        "% Positive Filtered Upside": (kal["filtered_upside"] > 0).mean() * 100,
    },
}

summary_df = pd.DataFrame(summary).T
display(summary_df)

if len(tri) > 0:
    print(f"\n🔗 Cross-Model Coverage: {len(tri):,} stocks in all 3 models")
    print(
        f"   Strong Bullish (3/3 agree): {(tri['agreement_score'] == 3).sum():,} ({(tri['agreement_score'] == 3).mean() * 100:.1f}%)")
    print(
        f"   Strong Bearish (0/3 agree): {(tri['agreement_score'] == 0).sum():,} ({(tri['agreement_score'] == 0).mean() * 100:.1f}%)")
    print(f"   MC ↔ Kalman correlation:    {tri[['expected_upside_pct', 'filtered_upside']].corr().iloc[0, 1]:.3f}")
    print(
        f"   MC ↔ Achievement corr:      {tri[['expected_upside_pct', 'expected_return_prob_weighted']].corr().iloc[0, 1]:.3f}")

print("\n✅ Expected Returns Analytics complete")


,Stocks Analyzed,Mean Expected Upside (%),Median Expected Upside (%),% Stocks with Positive Upside,Mean Prob Positive (%),Mean Prob-Weighted Return (%),Mean Achievement Prob,High Confidence Count,Mean Analyst Conviction (%),Mean Filtered Upside (%),Median Filtered Upside (%),Mean Signal Strength,% Positive Filtered Upside
Monte Carlo,5584.0,22.44,13.46,76.826648,74.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price Target Achievement,6405.0,NaN,NaN,NaN,NaN,38.39,0.609,1004.0,62.3,NaN,NaN,NaN,NaN
Kalman Filter,6403.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129.48,12.39,11.0,77.073247



🔗 Cross-Model Coverage: 5,584 stocks in all 3 models
   Strong Bullish (3/3 agree): 4,106 (73.5%)
   Strong Bearish (0/3 agree): 1,013 (18.1%)
   MC ↔ Kalman correlation:    0.979
   MC ↔ Achievement corr:      0.732

✅ Expected Returns Analytics complete
